# PC-FNO3D Tunisie — entraînement sur GPU (grille ERA5-Land)

Le principe de la v7 (FNO 3D + loss physique, cible u, v, w, ρ du sol à 1 km) sur **toute la
Tunisie** : capteurs = anémomètres des aéroports (METAR), vérité d'entraînement = ERA5-Land
(surface, 0,1°) + ERA5 (altitude). Un seul **Exécution → Tout exécuter**, sur GPU (T4, L4 ou A100).

**À déposer avant** (depuis ines-pc) :
1. `tunisie_donnees.zip` (≈ 0,9 Go, `python scripts/package_tunisie_colab.py`) → **une fois**
   dans `MyDrive/pcfno3d_tunisie/` (glisser-déposer sur drive.google.com) ;
2. `pcfno3d_code.zip` (`python scripts/package_for_colab.py`) → demandé **à chaque session**.

Tout ce qui doit survivre va sur le Drive (`MyDrive/pcfno3d_tunisie/`) : cache construit
(réutilisé aux sessions suivantes), checkpoints (toutes les 5 époques : une session coupée
reprend où elle s'était arrêtée), journaux, résultats de test.

## 1. Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Code (archive téléversée depuis ines-pc à chaque session)

In [ ]:
import json, shutil, subprocess, sys, time, zipfile
from pathlib import Path
from google.colab import files

MYDRIVE = Path("/content/drive/MyDrive")
TUN = MYDRIVE / "pcfno3d_tunisie"
TUN.mkdir(parents=True, exist_ok=True)
CIBLE = Path("/content/PFE")
DEPOSEE = Path("/content/pcfno3d_code.zip")   # panneau Fichiers -> Importer (session)
if DEPOSEE.exists():
    donnees = DEPOSEE.read_bytes()
else:
    print("Choisis pcfno3d_code.zip sur le PC (ou depose-le dans le panneau Fichiers) :")
    recu = files.upload()
    nom = next((n for n in recu if n.startswith("pcfno3d_code") and n.endswith(".zip")), None)
    if nom is None:
        raise FileNotFoundError(f"fichier recu : {list(recu)} - il faut pcfno3d_code.zip")
    donnees = recu[nom]
if CIBLE.exists():
    shutil.rmtree(CIBLE)
CIBLE.mkdir(parents=True)
(CIBLE / "code.zip").write_bytes(donnees)
with zipfile.ZipFile(CIBLE / "code.zip") as zf:
    zf.extractall(CIBLE)
tampon = CIBLE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

JOURNAUX = TUN / "journaux"
JOURNAUX.mkdir(parents=True, exist_ok=True)

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire, sortie affichee ET ecrite dans un journal du Drive (survit a une
    deconnexion) ; le notebook s'arrete si elle echoue."""
    t0 = time.time()
    journal = JOURNAUX / f"{time.strftime('%Y%m%dT%H%M%S')}_{nom}.log"
    with open(journal, "w", encoding="utf-8") as fj:
        proc = subprocess.Popen(commande, shell=True, cwd="/content/PFE", text=True,
                                env={**__import__("os").environ, "PYTHONPATH": "src"},
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        for ligne in proc.stdout:
            print(ligne, end="", flush=True)
            fj.write(ligne)
            fj.flush()
        code = proc.wait()
    print(f"
>>> {nom} : code {code} ({(time.time() - t0) / 60:.1f} min) - journal {journal}")
    if code != 0:
        raise RuntimeError(f"echec (code {code}) de : {commande}")

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print("pret")

## 3. Données brutes ERA5-Land + ERA5 + METAR (depuis le Drive)

In [ ]:
BRUT = Path("/content/tunisie_donnees")
ARCH = TUN / "tunisie_donnees.zip"
if not ARCH.exists():
    raise FileNotFoundError(f"depose tunisie_donnees.zip dans {TUN} (drive.google.com)")
if not BRUT.exists():
    with zipfile.ZipFile(ARCH) as zf:
        zf.extractall(BRUT)
print(len(list((BRUT / "era5").glob("*.nc"))), "fichiers ERA5 ;",
      len(list((BRUT / "metar").glob("*.csv"))), "fichiers METAR")

## 4. Jeu et cache (grille ERA5-Land)

Reconstruit à partir des données brutes (≈ 2 à 3 min par mois de données), puis gardé sur le
Drive (`cache_tunisie_land.zip`) : la session suivante le recopie au lieu de le reconstruire.
Mettre `RECONSTRUIRE = True` après avoir déposé de nouvelles données brutes.

In [ ]:
RECONSTRUIRE = False
LOCAL = Path("/content/tunisie_land")
SAUVE = TUN / "cache_tunisie_land.zip"
if SAUVE.exists() and not RECONSTRUIRE:
    if not (LOCAL / "cache" / "index.csv").exists():
        with zipfile.ZipFile(SAUVE) as zf:
            zf.extractall(LOCAL)
    print("cache recopie du Drive")
else:
    if RECONSTRUIRE and LOCAL.exists():
        shutil.rmtree(LOCAL)
    lancer("construire", "python -u scripts/tunisie_fno3d.py -c tunisie_fno3d_land_colab "
           f"--era5 {BRUT / 'era5'} --metar {BRUT / 'metar'} construire")
    tmp = SAUVE.with_suffix(".part")
    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
        for f in sorted(LOCAL.rglob("*")):
            if f.is_file():
                zf.write(f, f.relative_to(LOCAL))
    tmp.replace(SAUVE)
    print(f"cache garde sur le Drive : {SAUVE.stat().st_size / 1e9:.1f} Go")
import pandas as pd
print(pd.read_csv(LOCAL / "cache" / "index.csv")["split"].value_counts().to_string())

## 5. Résidu de masse de la vérité → `mass_ref`

In [ ]:
RUN = TUN / "runs" / "tunisie_fno3d_land_colab"
lancer("masse", "python -u scripts/tunisie_fno3d.py -c tunisie_fno3d_land_colab masse --instants 100")
MASS_REF = json.loads((RUN / "bilan_masse_verite.json").read_text())["mass_ref_conseille"]
print("mass_ref =", MASS_REF)

## 6. Entraînement

Reprise automatique : si la session a été coupée, relancer **Tout exécuter** reprend à la
dernière copie sur le Drive (au plus 5 époques perdues).

In [ ]:
lancer("entrainer", f"python -u -m pcfno3d -c tunisie_fno3d_land_colab --set loss.mass_ref={MASS_REF} train")

## 7. Test (2025) : erreurs par niveau et DSO

In [ ]:
lancer("evaluer", f"python -u -m pcfno3d -c tunisie_fno3d_land_colab --set loss.mass_ref={MASS_REF} evaluate")
for f in sorted(RUN.rglob("test_summary.json")):
    print(f)
    print(f.read_text()[:4000])
for f in sorted(RUN.rglob("test_par_niveau_synoptique.csv")):
    print(pd.read_csv(f).round(3).to_string())